---

Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida
definimos os caminhos das tabelas silver que serão utilizadas durante a execução.

#.
### Variáveis e suas utilizações
micro_batch_silver_path = define o caminho de leitura da tabela silver do micro batch

historico_silver_path = define o caminho de leitura e escrita da tabela silver do histórico

---

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

micro_batch_silver_path = "workspace.stocks.micro_batch_silver"
historico_silver_path = "workspace.stocks.silver"

---

Lemos a tabela silver do micro batch e identificamos a data mais recente disponível, filtrando apenas os registros do último pregão para consolidação no histórico.

---

In [0]:
df_micro = (
    spark.read.table(micro_batch_silver_path)
    .withColumn("data_pregao", sf.to_date("event_time"))
)

max_date = df_micro.select(sf.max("data_pregao")).collect()[0][0]

df_micro = df_micro.filter(sf.col("data_pregao") == sf.lit(max_date))

---

Calculamos o OHLCV diário a partir dos candles de um minuto do micro batch.

Utilizamos Window Functions para identificar o primeiro registro do dia como open e o último como close. O high, low e volume são calculados via agregação, pegando o maior valor, menor valor e soma do volume do dia respectivamente.

---

In [0]:
window_inicio = Window.partitionBy(["ticker"]).orderBy(sf.col("event_time").asc())
window_final = Window.partitionBy(["ticker"]).orderBy(sf.col("event_time").desc())

df_open = (df_micro.withColumn("rn", sf.row_number().over(window_inicio))
            .filter(sf.col("rn") == 1)
            ).select("ticker", "data_pregao", "open")

df_close = (df_micro.withColumn("rn", sf.row_number().over(window_final))
            .filter(sf.col("rn") == 1)
            ).select("ticker", "data_pregao", "close")

df_high_low_volume = (df_micro.groupBy("ticker", "data_pregao")
         .agg(
             sf.max("high").alias("high"),
             sf.min("low").alias("low"),
             sf.sum("volume").alias("volume")
         ))

---

Unimos os três dataframes calculados em um único dataframe diário, adicionamos as colunas de controle e calculamos a variação real e percentual do dia.

---

In [0]:
df_diario = (df_high_low_volume
          .join(df_open, ["ticker", "data_pregao"], "inner")
          .join(df_close, ["ticker", "data_pregao"], "inner")
          .withColumn("event_time", sf.to_timestamp("data_pregao"))
          .withColumn("ingestao_ts", sf.current_timestamp())
          .withColumn("fonte", sf.lit("append_historico"))
          .withColumn("week_year", sf.concat(sf.weekofyear("event_time"), sf.lit("-"), sf.year("event_time")))
          .withColumn("variacao_real", (sf.col("close") - sf.col("open")))
          .withColumn("variacao_percent", (sf.col("variacao_real") / sf.col("open")*100))
          ).select(
        "ticker",
        "event_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "fonte",
        "ingestao_ts",
        "week_year",
        "variacao_real",
        "variacao_percent"
    )

---

Lemos a tabela silver do histórico e realizamos um left_anti join para garantir que somente registros que ainda não existem no histórico sejam adicionados, evitando duplicidade de datas já consolidadas anteriormente.

---

In [0]:
df_historico = spark.read.table(historico_silver_path)

df_diario_novo = (df_diario.alias("novo")
                  .join(df_historico.select("ticker", "event_time").alias("hist"),
                        on = [sf.col("novo.ticker") == sf.col("hist.ticker"),
                              sf.col("novo.event_time") == sf.col("hist.event_time")],
                        how = "left_anti")
                  )

---

Exibimos uma amostra dos novos registros encontrados e salvamos em append
na tabela silver do histórico, consolidando os dados do pregão do dia.

---

In [0]:
df_diario_novo.show()
(
    df_diario_novo.write
    .format("delta")
    .mode("append")
    .saveAsTable(historico_silver_path)
)